# 🚀 Serverless SharePoint-to-GCS Synchronization Pipeline (V5.0)
## Interactive Google Colab Notebook & Walkthrough

Welcome to the interactive execution companion for the **V5.0 SharePoint-to-GCS Synchronization Pipeline**. This Colab notebook guides you step-by-step through configuration validation, deployment, execution, and troubleshooting.

---

### 🏛️ Architecture Overview
The sync pipeline utilizes a hybrid serverless design:
1. **Traversal Cloud Function (`Python Gen2`)**: Recursively crawls Microsoft Graph API to traverse nested folders/libraries and pre-render modern SharePoint site canvas layouts into static HTML files.
2. **Application Integration Parent Orchestrator**: Receives the traversed file manifest and loops over objects.
3. **Application Integration Child Worker**: Streams document bytes directly from SharePoint to Google Cloud Storage (GCS) and saves HTML rendered pages into the `pages/` directory.

---

### 📋 Prerequisites & Quick Colab Setup
Before starting, ensure that:
* You have uploaded your configured `parameters.json` file into this notebook's workspace session.
* Your Azure AD App Registration has `Sites.Read.All`, `Files.Read.All`, and `User.Read.All` scopes enabled.

---
## I. Colab Environment Setup & GCP Authentication

Run the cell below to authenticate your Google Cloud user account in Colab and set your default project from `parameters.json`.

In [ ]:
import json
import os

# 1. Authenticate user in Google Colab (if running inside Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Google Colab user authentication successful!")
except ImportError:
    print("ℹ️ Not running inside Colab. Using existing system Google Cloud credentials.")

# 2. Verify parameters.json exists in working directory
if not os.path.exists('parameters.json'):
    if os.path.exists('parameters.json.sample'):
        print("⚠️ 'parameters.json' not found. Creating a temporary copy from 'parameters.json.sample'...")
        import shutil
        shutil.copy('parameters.json.sample', 'parameters.json')
    else:
        raise FileNotFoundError("❌ Error: 'parameters.json' not found! Please upload your configured parameters.json file to continue.")

with open('parameters.json', 'r') as f:
    params = json.load(f)

project_id = params.get('CONFIG_ProjectId', '')
location = params.get('CONFIG_Location', 'asia-southeast1')

print(f"📌 Active GCP Project ID: {project_id}")
print(f"📌 Active Deployment Region: {location}")

# Set gcloud default project
!gcloud config set project {project_id}

---
## II. Pre-flight Validation & IAM Provisioning

### Step 0: Validate Configuration Parameters
Before running any deployment commands, validate that all parameter formats and live GCP resources (Project, Service Account, Storage Bucket, Secret Manager Secret, and Integration Connectors) are active and accessible.

In [ ]:
# Execute the automated parameter validation tool
!python3 util/validate_params.py

### Step 0.5: Provision Service Account and IAM Roles (Optional / First-time setup)
If the pre-flight check indicated that your Service Account or IAM roles do not exist yet, run this automated role-binding script. It reads `parameters.json` and provisions the custom Service Account and assigns required IAM bindings.

In [ ]:
!chmod +x prereq/sa-roles.sh
!./prereq/sa-roles.sh

---
## III. Pipeline Deployment

### Step 1: Export Environment Variables & Grant Secret Access
Load the configurations into the environment variables and grant the Cloud Function service account permission to read your M365 Azure AD Client Secret from GCP Secret Manager.

In [ ]:
import os

# Set environment variables for shell sub-processes
os.environ['PROJECT_ID'] = params.get('CONFIG_ProjectId', '')
os.environ['LOCATION'] = params.get('CONFIG_Location', '')
os.environ['SERVICE_ACCOUNT'] = params.get('CONFIG_Service_Account', '')
os.environ['DEVELOPER_PRINCIPAL'] = params.get('CONFIG_Developer_Group_Or_User', '')
os.environ['FUNCTION_NAME'] = params.get('CONFIG_CloudFunction_Name', 'yourorg-sharepoint-list-files')
os.environ['PARENT_INTEGRATION_NAME'] = params.get('CONFIG_Parent_Integration_Name', 'yourorg-sharepoint-gcs-parent')

secret_path = params.get('CONFIG_M365_Secret_Name', '')
secret_name = secret_path.split('/')[-1] if secret_path else ''
os.environ['SECRET_NAME'] = secret_name

print(f"🔑 Granting Secret Manager Accessor role on secret '{secret_name}' to '{os.environ['SERVICE_ACCOUNT']}'...")
!gcloud secrets add-iam-policy-binding "{os.environ['SECRET_NAME']}" \
    --member="serviceAccount:{os.environ['SERVICE_ACCOUNT']}" \
    --role="roles/secretmanager.secretAccessor" \
    --project="{os.environ['PROJECT_ID']}"

### Step 2: Deploy Traversal Cloud Function (`Python Gen2`)
Deploy the Cloud Function responsible for querying Microsoft Graph API, crawling folder hierarchies, and resolving modern site pages.

In [ ]:
!chmod +x deploy_cf.sh
!./deploy_cf.sh

### Step 3 & 3.5: Configure Cloud Run Invoker Roles & Run Diagnostic Test
Since Gen2 Cloud Functions run on Cloud Run, grant invocation permissions to the Service Account and your Developer User. Once bound, run a dry-run diagnostic test (`test_cf.py`) to confirm Graph API connectivity and count target SharePoint library files.

In [ ]:
# 1. Grant invoker rights to Cloud Scheduler Service Account
!gcloud run services add-iam-policy-binding "{os.environ['FUNCTION_NAME']}" \
    --region="{os.environ['LOCATION']}" \
    --member="serviceAccount:{os.environ['SERVICE_ACCOUNT']}" \
    --role="roles/run.invoker" \
    --project="{os.environ['PROJECT_ID']}"

# 2. Grant invoker rights to Developer Principal
dev_principal = os.environ.get('DEVELOPER_PRINCIPAL', '')
if 'group' in dev_principal or 'ggrp' in dev_principal or dev_principal.startswith('group:'):
    dev_member = f"group:{dev_principal.replace('group:', '')}"
else:
    dev_member = f"user:{dev_principal.replace('user:', '')}"

!gcloud run services add-iam-policy-binding "{os.environ['FUNCTION_NAME']}" \
    --region="{os.environ['LOCATION']}" \
    --member="{dev_member}" \
    --role="roles/run.invoker" \
    --project="{os.environ['PROJECT_ID']}"

# 3. Execute dry-run diagnostic traversal test
print("\n🔎 Running diagnostic Cloud Function test...")
!python3 test_cf.py

### Step 4: Parameterize & Deploy Integration Workflows
Compile the JSON templates (`child_workflow.json` and `parent_workflow.json`), dynamically inject your parameters, and publish both integrations to GCP Application Integration.

In [ ]:
!python3 deploy_workflows.py

---
## IV. Execution & Scheduling Operational Models

Choose and execute **ONE** of the three operational models below depending on how you wish to maintain your synchronization whitelist:

* **Option 1: Dynamic Remote Whitelist (Recommended)**: Reads target URLs live from `gs://BUCKET/config/target_urls.txt`.
* **Option 2: Full Traversal Sync**: Crawls and syncs all documents and rendered pages across the entire SharePoint site.
* **Option 3: Static Local Whitelist**: Syncs fixed URLs defined in `target_files.json`.

### Option 1: Dynamic Remote Whitelist via GCS (`Recommended`)
Upload your local `target_urls.txt` to Google Cloud Storage. Run on-demand or deploy an automated Cloud Scheduler cron job that checks GCS hourly.

In [ ]:
# 1. Upload local target_urls.txt to GCS configuration bucket
!chmod +x upload_gcs_targets.sh
!./upload_gcs_targets.sh

# 2. Run On-Demand Sync
print("\n🚀 Triggering dynamic on-demand sync runner...")
!python3 sync_gcs_dynamic.py

# 3. (Optional) Deploy automated Cloud Scheduler cron trigger
# !chmod +x deploy_scheduler_gcs_dynamic.sh
# !./deploy_scheduler_gcs_dynamic.sh

### Option 2: Full Traversal Sync
Crawl and synchronize **all** SharePoint library documents and rendered HTML site pages.

In [ ]:
# 1. Run full site traversal sync
!python3 sync_sharepoint_to_gcs.py

# 2. (Optional) Deploy full site traversal Cloud Scheduler cron
# !chmod +x deploy_scheduler.sh
# !./deploy_scheduler.sh

### Option 3: Static Local Whitelist (`target_files.json`)
Synchronize a static list of specific file URLs listed in `target_files.json`.

In [ ]:
# 1. Run targeted static whitelist sync
!python3 sync_specific_urls.py

# 2. (Optional) Deploy targeted Cloud Scheduler HTTP trigger
# !chmod +x deploy_scheduler_targeted.sh
# !./deploy_scheduler_targeted.sh

---
## V. Observability & Troubleshooting Diagnostics

Use the tools below to inspect workflow executions, Cloud Function logs, and Application Integration execution states.

In [ ]:
# 1. Check recent local diagnostic setup & cloud logs
print("📑 Recent log files in log/ directory:")
!ls -la log/

print("\n--- Latest Cloud Function execution log snippet ---")
!tail -n 20 log/cloud.log.* 2>/dev/null || echo "No cloud log found yet."

# 2. Check specific workflow execution status (Replace execution_uuid with your run ID)
execution_uuid = "YOUR_EXECUTION_UUID_HERE"
if execution_uuid != "YOUR_EXECUTION_UUID_HERE":
    !python3 check_application_integration_execution.py "{os.environ['PROJECT_ID']}" "{os.environ['LOCATION']}" "{os.environ['PARENT_INTEGRATION_NAME']}" "{execution_uuid}"
else:
    print("\nℹ️ To check specific execution detail, set 'execution_uuid' to your actual execution ID printed during sync.")

# 3. Read Cloud Function Traversal Logs via gcloud
print("\n--- Querying Google Cloud Logging for Traversal Cloud Function ---")
!gcloud logging read '(resource.type="cloud_run_revision" AND resource.labels.service_name="{os.environ["FUNCTION_NAME"]}") OR protoPayload.serviceName="run.googleapis.com"' \
    --project="{os.environ['PROJECT_ID']}" \
    --limit=10 \
    --format="table(timestamp, httpRequest.status, textPayload, protoPayload.status.message)"